# ComfyUI ワークフロー自動実行ノートブック（`maniol-elvish` 版 / 乱数プロンプト対応）

Google Colab 上で ComfyUI サーバーを起動し、`maniol-elvish` 向けに **乱数プロンプト生成サイクル** を回して画像生成・保存を行うためのノートブックです。  
コア処理は [`comfyui/runner/`](../runner/) にモジュール化されています。

### 主な特徴
- **`onnxruntime-gpu` クリーン導入**: CPU 版 `onnxruntime` による GPU 推論フォールバックを防止（[`pose-inspection-001.ipynb`](../../colab/snapshot-inspector/maniol-elvish/pose-inspection-001.ipynb) 準拠）
- **スロットベース乱数プロンプト生成**: 共通の Positive / Negative プロンプトに加え、`framing` / `orientation` / `angle` / `emotion` / `posture` の 5 要素を乱数で組み合わせ可能
- **要素ごとの細かな制御**: 各要素で乱数 ON/OFF（`enabled`）、採用確率（`probability`）、オフ時の固定タグ（`fixed`）、重み付き候補を指定可能
- **ループ回数指定**: `LOOP_COUNT` で乱数プロンプトの生成サイクル回数を指定し、各サイクルのプロンプトと出力画像の対応ログ（`generation_log.json`）も保存

### 事前準備
1. Colab のランタイムを **GPU（T4 / L4 / A100 等）** に設定
2. ComfyUI から **API 形式** でワークフロー JSON を書き出し（`Workflow -> Export (API)`）
3. 認証が必要なモデルを DL する場合は、Colab 左メニューの「シークレット（鍵アイコン）」に `HF_TOKEN` / `CIVITAI_API_KEY` を登録

## 1. GPU 確認 & Google Drive マウント

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. パス・設定の定義

Google Drive 上のモデルフォルダやワークフロー配置先を環境に合わせて調整してください。

In [ ]:
from pathlib import Path

# --- リポジトリ・キャラクター設定 ---
REPO_URL = "https://github.com/belre/stable-diffusion-pipelines.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/stable-diffusion-pipelines")
CHARACTER_NAME = "maniol-elvish"

# --- ComfyUI 本体・Drive パス設定 ---
COMFYUI_DIR = Path("/content/ComfyUI")
DRIVE_BASE = Path("/content/drive/MyDrive/ComfyUI")
DRIVE_MODELS_DIR = DRIVE_BASE / "models"                       # 自作 LoRA / VAE 等（loras/, checkpoints/, vae/）
DRIVE_WORKFLOWS_DIR = DRIVE_BASE / "workflows"                 # Drive 側に JSON を置く場合
DRIVE_INPUT_DIR = DRIVE_BASE / "input"                         # LoadImage 用入力画像
DRIVE_OUTPUT_DIR = DRIVE_BASE / "output" / CHARACTER_NAME      # 生成画像の保存先
DRIVE_MODELS_MANIFEST = DRIVE_BASE / "models.yaml"             # Drive に models.yaml を置く場合

# --- 追加インストールするカスタムノード（必要なものだけコメント解除・追記） ---
CUSTOM_NODES = [
    # "https://github.com/ltdrdata/ComfyUI-Manager.git",
]

# --- ComfyUI 起動オプション ---
COMFYUI_HOST = "127.0.0.1"
COMFYUI_PORT = 8188
COMFYUI_EXTRA_ARGS = []  # T4(15GB) で SDXL を使う場合は通常空で OK。VRAM が厳しい場合は ["--lowvram"]

print("CHARACTER_NAME   :", CHARACTER_NAME)
print("REPO_DIR         :", REPO_DIR)
print("COMFYUI_DIR      :", COMFYUI_DIR)
print("DRIVE_MODELS_DIR :", DRIVE_MODELS_DIR)
print("DRIVE_OUTPUT_DIR :", DRIVE_OUTPUT_DIR)

## 3. リポジトリ取得 & `runner` パッケージ準備

In [ ]:
import sys

# ローカル実行時は親ディレクトリを優先し、Colab 上ではリポジトリを clone/pull する
local_runner_root = Path("..").resolve()
if (local_runner_root / "runner" / "__init__.py").exists():
    RUNNER_ROOT = local_runner_root
else:
    if not REPO_DIR.exists():
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    RUNNER_ROOT = REPO_DIR / "comfyui"

!pip install -q -r {RUNNER_ROOT / 'requirements.txt'}

if str(RUNNER_ROOT) not in sys.path:
    sys.path.insert(0, str(RUNNER_ROOT))

from runner import (
    ComfyUIClient,
    ComfyUIServer,
    ModelSpec,
    PromptGenerator,
    PromptSlot,
    check_workflow_models,
    download_models,
    generate_extra_model_paths_yaml,
    load_model_manifest,
    load_workflow,
)
print("RUNNER_ROOT:", RUNNER_ROOT)
print("runner モジュールの読み込みに成功しました。")

## 4. ComfyUI 本体のセットアップ + `onnxruntime-gpu` 導入 & SDXL モデル準備

1. ComfyUI 本体を `/content/ComfyUI` にインストール
2. CPU 版 `onnxruntime` の混入を防ぐため、`onnxruntime` / `onnxruntime-gpu` を一度アンインストールしてから `onnxruntime-gpu` をクリーンインストール（[`pose-inspection-001.ipynb`](../../colab/snapshot-inspector/maniol-elvish/pose-inspection-001.ipynb) 準拠）
3. `extra_model_paths.yaml` を生成し、Google Drive 上の自作 LoRA 等をコピーなしで認識させる
4. 高速 DL 用の `aria2` を導入し、公開 SDXL モデルを `/content/ComfyUI/models/` へ自動ダウンロード

In [ ]:
import subprocess

if not COMFYUI_DIR.exists():
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFYUI_DIR}
else:
    print("ComfyUI は既にクローン済みです:", COMFYUI_DIR)

# ComfyUIServer と同じ Python インタプリタ (sys.executable) に依存関係を確実に導入する
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFYUI_DIR / "requirements.txt")],
    check=True,
)

# カスタムノードの導入（指定がある場合）
custom_nodes_dir = COMFYUI_DIR / "custom_nodes"
for repo_url in CUSTOM_NODES:
    node_name = repo_url.rstrip("/").split("/")[-1].removesuffix(".git")
    target_dir = custom_nodes_dir / node_name
    if not target_dir.exists():
        !git clone {repo_url} {target_dir}
        req = target_dir / "requirements.txt"
        if req.is_file():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

# ComfyUI やカスタムノードが CPU 版 onnxruntime を引き込んで GPU 推論を潰すのを防ぐため、
# 両方を完全に除外してから onnxruntime-gpu のみをクリーンインストールする
# (参照: colab/snapshot-inspector/maniol-elvish/pose-inspection-001.ipynb)
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu"],
    check=False,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "onnxruntime-gpu"],
    check=True,
)

import onnxruntime as ort

if hasattr(ort, "preload_dlls"):
    try:
        ort.preload_dlls()
    except Exception as exc:
        print(f"[INFO] ort.preload_dlls() skipped: {exc}")

print("ONNX Runtime version   :", ort.__version__)
print("ONNX Runtime providers :", ort.get_available_providers())

# Google Drive のモデルフォルダを ComfyUI に認識させる（自作 LoRA 等向け）
EXTRA_MODEL_PATHS = None
if DRIVE_MODELS_DIR.is_dir():
    EXTRA_MODEL_PATHS = generate_extra_model_paths_yaml(
        drive_models_dir=DRIVE_MODELS_DIR,
        output_path=COMFYUI_DIR / "extra_model_paths_colab.yaml",
    )
    print("extra_model_paths を生成しました:", EXTRA_MODEL_PATHS)
else:
    print(f"[INFO] {DRIVE_MODELS_DIR} が見つかりません。自作 LoRA 等を使う場合は Drive に作成してください。")

In [ ]:
# --- 公開 SDXL モデルの自動ダウンロード（Hugging Face / Civitai） ---
!apt-get -qq update && apt-get -qq install -y aria2

# Colab Secrets からトークンを取得（未設定でもエラーにならない）
def get_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

HF_TOKEN = get_secret("HF_TOKEN")
CIVITAI_API_KEY = get_secret("CIVITAI_API_KEY")
print("HF_TOKEN        :", "設定あり" if HF_TOKEN else "未設定")
print("CIVITAI_API_KEY :", "設定あり" if CIVITAI_API_KEY else "未設定")

# 1) Drive に models.yaml がある場合はそこから読み込む
model_specs: list[ModelSpec] = []
if DRIVE_MODELS_MANIFEST.is_file():
    model_specs.extend(load_model_manifest(DRIVE_MODELS_MANIFEST))
    print(f"{DRIVE_MODELS_MANIFEST} から {len(model_specs)} 件のモデル定義を読み込みました。")

# 2) ノートブック上で直接指定したいモデルがあればここに追記（必要なものだけコメント解除）
INLINE_MODELS: list[ModelSpec] = [
    # 例: Animagine XL 3.1 (Hugging Face)
    # ModelSpec(
    #     subdir="checkpoints",
    #     filename="animagine-xl-3.1.safetensors",
    #     hf_repo="cagliostrolab/animagine-xl-3.1",
    #     hf_file="animagine-xl-3.1.safetensors",
    # ),
]
model_specs.extend(INLINE_MODELS)

if model_specs:
    download_models(
        model_specs,
        comfyui_dir=COMFYUI_DIR,
        hf_token=HF_TOKEN,
        civitai_token=CIVITAI_API_KEY,
        drive_models_dir=DRIVE_MODELS_DIR if DRIVE_MODELS_DIR.is_dir() else None,
    )
else:
    print("自動 DL 対象のモデルは未指定です（Drive 側モデルのみを使用します）。")

## 5. ComfyUI サーバー起動（バックグラウンド）

In [ ]:
server = ComfyUIServer(
    comfyui_dir=COMFYUI_DIR,
    host=COMFYUI_HOST,
    port=COMFYUI_PORT,
    extra_args=COMFYUI_EXTRA_ARGS,
    extra_model_paths_config=EXTRA_MODEL_PATHS,
)
client = server.start(wait_ready=True, timeout=180.0)
print("ComfyUI サーバー起動完了:", server.base_url)

## 6. 乱数プロンプト設定 & 生成サイクルのプレビュー

共通の Positive / Negative プロンプトに加え、以下の 5 要素を乱数で生成します。
- `framing`（画角・ショットサイズ）
- `orientation`（顔・体の向き・視線）
- `angle`（カメラ仰角・アングル）
- `emotion`（表情）
- `posture`（姿勢・ポーズ）

各要素では以下を指定できます。
- `enabled`: `True` で乱数有効 / `False` で乱数オフ
- `probability`: 乱数採用確率（`0.0`〜`1.0`）。`0.0` にすると実質オフ
- `fixed`: `enabled=False` のときに固定で入れたいタグ（不要なら `None`）
- `choices`: 候補リスト `["tag1", "tag2"]` または重み付き辞書 `{"tag1": 0.7, "tag2": 0.3}`

In [ ]:
# --- 共通 Positive / Negative プロンプト（maniol-elvish 固定要素） ---
COMMON_POSITIVE_PROMPT = (
    "maniol-elvish, 1girl, solo, green eyes, short hair, braid, ahoge, "
    "lace-up, green tunic, simple background"
)
COMMON_NEGATIVE_PROMPT = (
    "lowres, bad anatomy, bad hands, extra hair, duplicate, worst quality, low quality, "
    "watermark, signature, text, large breasts, huge breasts, long hair, twin braids, "
    "necklace, gloves, thigh strap, holster, arm band"
)

# --- 5 要素の乱数設定（各要素で enabled / probability / fixed / choices を調整可能） ---
PROMPT_SLOTS = {
    "framing": {
        "enabled": True,
        "probability": 1.0,
        "fixed": None,
        "choices": [
            "portrait",
            "upper body",
            "cowboy shot",
            "full body",
        ],
    },
    "orientation": {
        "enabled": True,
        "probability": 0.8,
        "fixed": None,
        "choices": [
            "looking at viewer",
            "looking away",
            "looking to the side",
            "looking back",
            "profile",
            "from side",
            "from behind",
            "straight-on",
        ],
    },
    "angle": {
        "enabled": True,
        "probability": 0.4,  # 60% は未指定（自然な eye-level）にする例
        "fixed": None,
        "choices": [
            "from above",
            "from below",
            "dutch angle",
        ],
    },
    "emotion": {
        "enabled": True,
        "probability": 0.7,
        "fixed": None,
        "choices": [
            "smile",
            "open mouth",
            "closed mouth",
            "serious",
            "expressionless",
            "surprised",
            "blush",
            "sad",
            "angry",
            "closed eyes",
        ],
    },
    "posture": {
        "enabled": True,
        "probability": 0.7,
        "fixed": None,
        "choices": [
            "standing",
            "sitting",
            "walking",
            "running",
            "hand on hip",
            "crossed arms",
            "leaning forward",
            "hands behind back",
            "head tilt",
            "arm up",
        ],
    },
}

# --- 乱数サイクルのループ設定 ---
LOOP_COUNT = 5          # 乱数プロンプトを何サイクル生成・実行するか
PROMPT_RNG_SEED = None  # 再現したい場合は整数を指定（毎回変える場合は None）

prompt_generator = PromptGenerator(
    common_positive=COMMON_POSITIVE_PROMPT,
    common_negative=COMMON_NEGATIVE_PROMPT,
    slots=PROMPT_SLOTS,
    seed=PROMPT_RNG_SEED,
)
generated_prompts = prompt_generator.generate_cycle(LOOP_COUNT)

print(f"=== 生成プロンプト プレビュー（{LOOP_COUNT} サイクル）===")
for idx, gp in enumerate(generated_prompts, start=1):
    slots_summary = ", ".join(f"{k}={v or '-'}" for k, v in gp.selected_slots.items())
    print(f"[{idx:03d}] ({slots_summary})")
    print(f"      POS: {gp.positive}")

## 7. SDXL プリセット選択 & ワークフロー JSON のループ実行

Section 6 で生成した `LOOP_COUNT` 件の乱数プロンプトを 1 サイクルずつ順番にワークフローへ適用して画像を生成します。  
生成完了後、各サイクルのプロンプト・選択スロット・出力ファイル一覧を `generation_log.json` へ保存します。

In [ ]:
import json
from datetime import datetime

# リポジトリ内の comfyui/workflows を使うか、Drive の workflows を使うか選択
WORKFLOWS_SOURCE_DIR = DRIVE_WORKFLOWS_DIR if DRIVE_WORKFLOWS_DIR.is_dir() else (RUNNER_ROOT / "workflows")
INPUT_DIR = DRIVE_INPUT_DIR if DRIVE_INPUT_DIR.is_dir() else (RUNNER_ROOT / "input")
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("/content/comfyui_output") / CHARACTER_NAME / RUN_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- SDXL プリセット & パラメータ上書き設定 ---
PRESET_PATH = RUNNER_ROOT / "presets" / "sdxl_default.yaml"  # 使わない場合は None
RANDOMIZE_SEED = True     # True にすると実行ごとにシード値をランダム化
FIXED_SEED = None         # 固定シードを使いたい場合は整数を指定（RANDOMIZE_SEED より優先）
REPEAT_PER_PROMPT = 1     # 同一ランダムプロンプトあたりの繰り返し実行回数
OVERRIDES = {
    # Maniol Elvish V4 LoRA を使う場合の例
    "class:LoraLoader.lora_name": "maniol-elvish-v4.safetensors",
    "class:SaveImage.filename_prefix": "maniol-elvish-v4",
    # 例: "class:CheckpointLoaderSimple.ckpt_name": "animagine-xl-3.1.safetensors",
    # 例: "class:KSampler.steps": 28,
}

# --- 事前チェック: ワークフローが参照するモデルがローカル/Drive に揃っているか確認 ---
for wf_file in sorted(WORKFLOWS_SOURCE_DIR.glob("*.json")):
    check = check_workflow_models(
        load_workflow(wf_file),
        comfyui_dir=COMFYUI_DIR,
        drive_models_dir=DRIVE_MODELS_DIR if DRIVE_MODELS_DIR.is_dir() else None,
    )
    if check["missing"]:
        print(f"[WARN] {wf_file.name} で未配置のモデルがあります: {check['missing']}")
    else:
        print(f"[OK] {wf_file.name}: 参照モデル確認済み ({len(check['found'])} 件)")

def on_progress(value: int, max_value: int, node: str | None) -> None:
    print(f"\r  [step {value}/{max_value}] node={node}", end="", flush=True)

results = []
generation_log = []

for cycle_idx, gp in enumerate(generated_prompts, start=1):
    slots_summary = ", ".join(f"{k}={v or '-'}" for k, v in gp.selected_slots.items())
    print(f"\n[Cycle {cycle_idx}/{len(generated_prompts)}] {slots_summary}")
    cycle_results = client.run_directory(
        workflows_dir=WORKFLOWS_SOURCE_DIR,
        output_dir=OUTPUT_DIR,
        preset=PRESET_PATH,
        overrides=OVERRIDES or None,
        randomize_seed=RANDOMIZE_SEED,
        seed=FIXED_SEED,
        positive_prompt=gp.positive,
        negative_prompt=gp.negative,
        input_dir=INPUT_DIR if INPUT_DIR.is_dir() else None,
        repeat=REPEAT_PER_PROMPT,
        prefix=f"cycle{cycle_idx:03d}_",
        progress_callback=on_progress,
    )
    print()
    results.extend(cycle_results)
    generation_log.append({
        "cycle": cycle_idx,
        "selected_slots": gp.selected_slots,
        "positive_prompt": gp.positive,
        "negative_prompt": gp.negative,
        "output_files": [f.name for r in cycle_results for f in r.output_files],
        "elapsed_seconds": round(sum(r.elapsed_seconds for r in cycle_results), 2),
    })

LOG_PATH = OUTPUT_DIR / "generation_log.json"
LOG_PATH.write_text(
    json.dumps(
        {
            "character": CHARACTER_NAME,
            "run_tag": RUN_TAG,
            "loop_count": len(generated_prompts),
            "repeat_per_prompt": REPEAT_PER_PROMPT,
            "prompt_rng_seed": PROMPT_RNG_SEED,
            "cycles": generation_log,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(f"\n実行完了: {len(results)} ジョブ / ログ保存先: {LOG_PATH}")
for r in results:
    print(f"- {r.workflow_path.name if r.workflow_path else r.prompt_id}: {len(r.output_files)} file(s), {r.elapsed_seconds:.1f}s")

## 8. 生成画像のプレビュー & Google Drive への保存

In [ ]:
from IPython.display import display
from PIL import Image

MAX_PREVIEW_IMAGES = 12  # 大量ループ時のブラウザ負荷を抑えるためプレビュー上限を設定

generated_files = [f for r in results for f in r.output_files]
if not generated_files:
    print("生成された画像がありません。workflows フォルダに API 形式 JSON があるか確認してください。")
else:
    print(f"生成画像数: {len(generated_files)} 枚（先頭 {min(len(generated_files), MAX_PREVIEW_IMAGES)} 枚を表示）")
    for img_path in generated_files[:MAX_PREVIEW_IMAGES]:
        print(img_path.name)
        if img_path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}:
            display(Image.open(img_path))

In [ ]:
import shutil

if generated_files:
    drive_run_dir = DRIVE_OUTPUT_DIR / RUN_TAG
    drive_run_dir.mkdir(parents=True, exist_ok=True)
    for img_path in generated_files:
        dst = drive_run_dir / img_path.name
        shutil.copy2(img_path, dst)
        print("Drive に保存:", dst)
    if LOG_PATH.is_file():
        dst_log = drive_run_dir / LOG_PATH.name
        shutil.copy2(LOG_PATH, dst_log)
        print("ログを保存   :", dst_log)

## 9. サーバー停止（作業終了時）

In [ ]:
server.stop()
print("ComfyUI サーバーを停止しました。")

### VS Code 上でのランタイム終了手順
作業が終わったら、GPUランタイムの課金・リソース消費を防ぐため、必ずランタイムを切断・削除してください。

1. VS Code 上部のコマンドパレット（`Ctrl + Shift + P` / macOS は `Cmd + Shift + P`）を開く
2. **`Colab: Remove Server`** を入力して実行する
3. 確認ダイアログが出たら対象の Colab ランタイムを削除して終了する